# Enron: hyperedge communities, centrality and figures
Self-contained Google Colab notebook. Upload this **.ipynb** using File → Upload notebook, then choose Runtime → Run all.

Sources: [hyperedge percolation paper](https://doi.org/10.1038/s41598-025-19974-9); [Enron dataset and citation](https://www.cs.cornell.edu/~arb/data/email-Enron/); [Walmart dataset and citation](https://www.cs.cornell.edu/~arb/data/walmart-trips/).

Both notebooks were executed on the supplied archives before delivery. Computational completion is not evidence of scientific novelty or superiority. See each section's scope notes before using outputs in a paper.


## 1. Load the dataset
Run all cells. Upload the raw archive when asked (Enron: email-Enron.tar.gz; Walmart: walmart-trips.zip). No GPU or results ZIP is needed. Four parameter settings are evaluated; PRIMARY=(4,3) is illustrative, not tuned for high correlations. Both packages run independently of prior notebooks.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import time, json, tarfile, zipfile, hashlib, platform
import numpy as np
import pandas as pd
import scipy, networkx as nx
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, shortest_path
from scipy.sparse.linalg import eigsh
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

DATASET = 'Enron'  # replaced by notebook builder
SEED=42
SETTINGS=[(3,2),(4,2),(4,3),(5,4)]
PRIMARY=(4,3)
CORE_SIZE=250
OUT=Path(DATASET+'_Community_Results'); OUT.mkdir(exist_ok=True)
timings=[]
filename='email-Enron.tar.gz' if DATASET=='Enron' else 'walmart-trips.zip'
archive=Path(filename)
if not archive.exists():
    local=Path('upload')/filename
    if local.exists(): archive=local
    else:
        from google.colab import files
        uploaded=files.upload()
        suffix='.tar.gz' if DATASET=='Enron' else '.zip'
        matches=[p for p in uploaded if p.endswith(suffix)]
        if len(matches)!=1: raise ValueError('Upload only '+filename)
        archive=Path(matches[0])
archive_hash=hashlib.sha256(archive.read_bytes()).hexdigest()



## 2. Static unique-set representation
Repeated records are collapsed into unique unordered hyperedges. Record frequency is recorded internally but does not weight community detection or centralities. Node IDs are preserved. Enron labels are email addresses; Walmart labels are departments, not product names. Each incidence is stored in an integer sparse matrix.

In [ ]:
t=time.perf_counter()
if DATASET=='Enron':
    with tarfile.open(archive) as z:
        def read(part):
            m=next(m for m in z.getmembers() if m.isfile() and m.name.endswith('email-Enron-'+part+'.txt'))
            return z.extractfile(m).read().decode()
        sizes=list(map(int,read('nverts').split())); flat=list(map(int,read('simplices').split()))
        times=list(map(int,read('times').split()))
        labels={int(l.split(maxsplit=1)[0]):l.split(maxsplit=1)[1] for l in read('node-labels').splitlines() if l.strip()}
    assert sum(sizes)==len(flat) and len(sizes)==len(times)
    raw=[];offset=0
    for size in sizes:raw.append(tuple(sorted(set(flat[offset:offset+size]))));offset+=size
else:
    with zipfile.ZipFile(archive) as z:
        def read(part):
            name=next(n for n in z.namelist() if n.endswith(part+'-walmart-trips.txt'))
            return z.read(name).decode()
        raw=[tuple(sorted(set(map(int,l.split(','))))) for l in read('hyperedges').splitlines() if l.strip()]
        departments=read('label-names').splitlines()
        ids=list(map(int,read('node-labels').split()))
        labels={i+1:departments[c-1] for i,c in enumerate(ids)}
frequency=Counter(raw);edges=sorted(frequency); vertices=sorted(set().union(*map(set,edges)))
n=len(vertices);index={v:i for i,v in enumerate(vertices)}
rows=[];cols=[]
for j,e in enumerate(edges):
    for v in e: rows.append(index[v]);cols.append(j)
H=csr_matrix((np.ones(len(rows),dtype=np.int32),(rows,cols)),shape=(n,len(edges)))
edge_sizes=np.asarray(H.sum(axis=0)).ravel();degree=np.asarray(H.sum(axis=1)).ravel()
timings.append({'operation':'parse_and_incidence','seconds':time.perf_counter()-t})
print(DATASET,':',n,'vertices,',len(raw),'records,',len(edges),'unique unordered hyperedges')
pd.DataFrame({'node_id':vertices,'label':[labels[v] for v in vertices],'full_unique_hyperdegree':degree}).to_csv(OUT/'full_vertex_labels_degree.csv',index=False)



## 3. Full-data hyperedge percolation
Retain |e|≥k and join hyperedges when their intersection has size ≥s. Connected hyperedge components produce vertex communities; duplicates and contained communities are removed. This is the large-hyperedge/absolute-intersection branch of Kovács et al. (2025). Singleton retained hyperedges can constitute communities. Memberships may overlap; vertices can be unassigned.

The sparse overlap matrix is calculated in blocks, avoiding a dense all-hyperedge matrix. Full-data community detection does not imply full-data dangling computation. Exported membership files cover all input vertices for each parameter setting.

In [ ]:
def detect(k,s,block_size=256):
    selected=np.flatnonzero(edge_sizes>=k);m=len(selected)
    if m==0:return [],0
    h=H[:,selected].astype(np.int32).tocsr()
    # A union-find avoids storing the complete hyperedge-intersection graph.
    parent=np.arange(m);rank=np.zeros(m,dtype=np.int8)
    def find(a):
        while parent[a]!=a:parent[a]=parent[parent[a]];a=parent[a]
        return a
    def union(a,b):
        a=find(a);b=find(b)
        if a==b:return
        if rank[a]<rank[b]:a,b=b,a
        parent[b]=a
        if rank[a]==rank[b]:rank[a]+=1
    for start in range(0,m,block_size):
        intersection=(h[:,start:start+block_size].T@h).tocoo()
        ok=(intersection.data>=s)&(intersection.col>intersection.row+start)
        for a,b in zip(intersection.row[ok]+start,intersection.col[ok]):union(int(a),int(b))
    components=defaultdict(set)
    for j,edge_id in enumerate(selected):components[find(j)].update(edges[edge_id])
    candidates=sorted(set(map(frozenset,components.values())),key=lambda c:(-len(c),tuple(sorted(c))))
    # Remove strict subsets using postings of already-kept larger communities.
    kept=[];postings=defaultdict(set)
    for c in candidates:
        anchor=min(c,key=lambda v:len(postings[v]))
        if any(c<=kept[j] for j in postings[anchor]):continue
        j=len(kept);kept.append(c)
        for v in c:postings[v].add(j)
    return kept,m

detected={};sensitivity=[]
for k,s in SETTINGS:
    t=time.perf_counter();cs,retained=detect(k,s);elapsed=time.perf_counter()-t
    memberships=[[] for _ in vertices]
    for j,c in enumerate(cs,1):
        for v in c:memberships[index[v]].append(j)
    counts=np.array(list(map(len,memberships)))
    detected[k,s]=(cs,memberships,counts)
    sensitivity.append({'k':k,'s':s,'communities':len(cs),'retained_hyperedges':retained,
        'assigned_vertices':int((counts>0).sum()),'overlap_vertices':int((counts>1).sum()),
        'unassigned_vertices':int((counts==0).sum()),'largest_community':max(map(len,cs),default=0),'seconds':elapsed})
    pd.DataFrame({'node_id':vertices,'community_ids':[';'.join(map(str,x)) for x in memberships],
                  'community_count':counts}).to_csv(OUT/f'memberships_k{k}_s{s}.csv',index=False)
    print('Completed',k,s,':',len(cs),'communities;',round(elapsed,2),'seconds',flush=True)
sensitivity=pd.DataFrame(sensitivity);display(sensitivity);sensitivity.to_csv(OUT/'community_sensitivity.csv',index=False)
communities,memberships,counts=detected[PRIMARY]
pd.DataFrame({'community_id':range(1,len(communities)+1),'size':list(map(len,communities))}).to_csv(OUT/'community_sizes.csv',index=False)



## 4. Centralities and community relationships
**Enron:** all five centralities are calculated on the full unique-set hypergraph.

**Walmart:** select the 250 highest full-unique-hyperdegree vertices (ties by ID). Intersect each original edge with that vertex set, discard sizes below 2, and deduplicate. Recompute ALL FIVE centralities on this same subset hypergraph. Hyperdegree is not mixed with degree from the full network. This degree-selected subset is biased and cannot support conclusions about full-network dangling rankings. These results can differ from the earlier notebook, which selected using repeated-record degrees.

Memberships are from the full hypergraph. Thus the Walmart relationship is explicitly between subset centrality and full-data community membership of those selected vertices. Report it as exploratory, alongside sensitivity and assigned-only correlations.

Closeness uses unweighted two-section paths and disconnected-network correction. Betweenness counts vertex paths, not distinct hyperedge-labelled paths. Eigenvector uses HHᵀ−D, max-normalized, and is left missing if the leading eigenvalue is degenerate. On disconnected networks it can concentrate on the component with highest spectral radius.

Dangling: Φ=Σ(i<j)1/d(i,j), unreachable contributions zero; DC(v)=[Φ−Φ(H⁻ᵛ)]/Φ. Remove v from incident edges, retaining other vertices and isolating v. This equals unweighted two-section vertex-isolation efficiency loss. It does not distinguish different hyperedge groupings having the same two-section.

In [ ]:
t=time.perf_counter()
if DATASET=='Enron':
    selected_vertices=vertices;analysis_edges=edges;scope='full hypergraph'
else:
    ranked=sorted(vertices,key=lambda v:(-degree[index[v]],v))
    selected_vertices=sorted(ranked[:CORE_SIZE]);selected_set=set(selected_vertices)
    analysis_edges=sorted(set(tuple(v for v in e if v in selected_set) for e in edges))
    analysis_edges=[e for e in analysis_edges if len(e)>=2]
    scope=f'top-{len(selected_vertices)} full-unique-hyperdegree vertices; deduplicated shrunken edges of size >=2'
local_index={v:i for i,v in enumerate(selected_vertices)};nn=len(selected_vertices)
rr=[];cc=[]
for j,e in enumerate(analysis_edges):
    for v in e:rr.append(local_index[v]);cc.append(j)
HH=csr_matrix((np.ones(len(rr),dtype=np.int32),(rr,cc)),shape=(nn,len(analysis_edges)))
W=(HH@HH.T).astype(float);W.setdiag(0);W.eliminate_zeros();A=W.copy();A.data[:]=1
G=nx.from_scipy_sparse_array(A)
timings.append({'operation':'centrality_network_construction','seconds':time.perf_counter()-t})
def timed(name,fn):
    t=time.perf_counter();x=fn();timings.append({'operation':name,'seconds':time.perf_counter()-t});return x
hd=timed('hyperdegree_same_analysis_hypergraph',lambda:np.asarray(HH.sum(axis=1)).ravel())
cl=timed('closeness_exact',lambda:nx.closeness_centrality(G,wf_improved=True))
bt=timed('betweenness_exact',lambda:nx.betweenness_centrality(G,normalized=True))
def eig():
    vals,vecs=eigsh(W,k=2,which='LA',v0=np.ones(nn),tol=1e-10)
    order=np.argsort(vals);lam=vals[order[-1]];gap=lam-vals[order[-2]]
    x=vecs[:,order[-1]];x=x if x.sum()>0 else -x
    residual=np.linalg.norm(W@x-lam*x)/max(abs(lam),1)
    if gap<1e-8*max(abs(lam),1):return np.full(nn,np.nan)
    assert residual<1e-6 and x.min()>-1e-6
    x=np.maximum(x,0);x[x<1e-12]=0;return x/x.max()
ev=timed('eigenvector_exact_network',eig)
def phi(a):
    d=shortest_path(a,directed=False,unweighted=True,method='D');u=d[np.triu_indices(len(d),1)]
    return float((1/u[np.isfinite(u)&(u>0)]).sum())
def dangling():
    baseline=phi(A);values=[]
    for i in range(nn):
        a=A.tolil();a[i,:]=0;a[:,i]=0;a=a.tocsr();a.eliminate_zeros()
        values.append((baseline-phi(a))/baseline if baseline else 0)
    return np.array(values)
dc=timed('dangling_exact',dangling)
assert np.isfinite(dc).all() and dc.min()>=-1e-10
results=pd.DataFrame({'node_id':selected_vertices,'label':[labels[v] for v in selected_vertices],
    'Hyperdegree':hd,'Closeness':[cl[i] for i in range(nn)],'Betweenness':[bt[i] for i in range(nn)],
    'Eigenvector':ev,'Dangling':dc,'full_hypergraph_community_count':[counts[index[v]] for v in selected_vertices],
    'full_hypergraph_community_ids':[';'.join(map(str,memberships[index[v]])) for v in selected_vertices]})
results.to_csv(OUT/'centralities_and_memberships.csv',index=False)
display(results.sort_values('Dangling',ascending=False).head(10))
def rho(x,y):
    return float(spearmanr(x,y).statistic) if len(x)>2 and np.ptp(x)>0 and np.ptp(y)>0 else np.nan
associations=[]
for k,s in SETTINGS:
    cnt=np.array([detected[k,s][2][index[v]] for v in selected_vertices]);assigned=cnt>0
    associations.append({'k':k,'s':s,'centrality_vertices':nn,'assigned_centrality_vertices':int(assigned.sum()),
        'rho_all':rho(dc,cnt),'rho_assigned_only':rho(dc[assigned],cnt[assigned])})
pd.DataFrame(associations).to_csv(OUT/'dangling_membership_associations.csv',index=False)



## 5. A readable picture of REAL selected hyperedges
Select six original hyperedges of size 3–6 by a reproducible connected expansion around high-degree vertices, capped at 26 displayed vertices. Every displayed hyperedge is complete; this is a selected edge subhypergraph, not the entire network or a representative sample.

Colored contours denote hyperedges. Node colors denote full-data community-membership status (single, multiple or unassigned), NOT distinct community IDs. The membership key supplies actual community IDs. Shapes use member disks joined by tubes with nonmember disks removed; regions may have holes or disconnected pieces to preserve node-center memberships. Geometric overlap away from node centers carries no membership meaning. Exact membership tables accompany every picture.

These are structural figures; no GNN, feature-fusion or message-passing model is claimed.

In [ ]:
def select_picture_edges(max_edges=6,max_vertices=26):
    eligible=[j for j,e in enumerate(edges) if 3<=len(e)<=6]
    # Deterministic connected expansion around high full hyperdegree.
    eligible.sort(key=lambda j:(-sum(degree[index[v]] for v in edges[j]),j))
    selected=[];seen=set()
    for j in eligible:
        e=set(edges[j])
        if selected and not(e&seen):continue
        if selected and not(e-seen):continue
        if len(seen|e)>max_vertices:continue
        selected.append(j);seen|=e
        if len(selected)==max_edges:break
    return selected
picture_edges=select_picture_edges()
picture_vertices=sorted(set().union(*(set(edges[j]) for j in picture_edges)))
P=nx.Graph();P.add_nodes_from(picture_vertices)
for j in picture_edges:P.add_edges_from(combinations(edges[j],2))
pos=nx.spring_layout(P,seed=SEED,k=1.1/np.sqrt(len(P)),iterations=300)
# Radial edge regions use member disks and minimum-spanning-tree tubes.
# Nonmember disks are cut out so enclosed node centers encode exact membership.
xy=np.array(list(pos.values()));extent=[xy[:,0].min()-.3,xy[:,0].max()+.3,xy[:,1].min()-.3,xy[:,1].max()+.3]
xx,yy=np.meshgrid(np.linspace(extent[0],extent[1],650),np.linspace(extent[2],extent[3],650))
def field(members):
    f=np.full(xx.shape,np.inf)
    complete=nx.Graph();complete.add_nodes_from(members)
    for a,b in combinations(members,2):complete.add_edge(a,b,weight=float(np.linalg.norm(pos[a]-pos[b])))
    for a,b in nx.minimum_spanning_tree(complete).edges():
        p,q=pos[a],pos[b];delta=q-p
        u=np.clip(((xx-p[0])*delta[0]+(yy-p[1])*delta[1])/np.dot(delta,delta),0,1)
        distance=np.sqrt((xx-p[0]-u*delta[0])**2+(yy-p[1]-u*delta[1])**2)
        f=np.minimum(f,distance-.065)
    for v in members:f=np.minimum(f,np.sqrt((xx-pos[v][0])**2+(yy-pos[v][1])**2)-.09)
    for v in set(picture_vertices)-set(members):f=np.maximum(f,.055-np.sqrt((xx-pos[v][0])**2+(yy-pos[v][1])**2))
    return f
plt.rcParams.update({'font.size':10})
fig,(ax,tableax)=plt.subplots(1,2,figsize=(14,7),gridspec_kw={'width_ratios':[1.35,1]})
colors=plt.get_cmap('tab10').colors
for number,j in enumerate(picture_edges):
    f=field(edges[j]);color=colors[number%10]
    ax.contourf(xx,yy,f,levels=[-100,0],colors=[color],alpha=.13)
    ax.contour(xx,yy,f,levels=[0],colors=[color],linewidths=1.5)
for v in picture_vertices:
    count=counts[index[v]]; color='#c67419' if count>1 else ('#2b728b' if count==1 else '#9c9c9c')
    ax.scatter(*pos[v],s=360,c=color,edgecolors='white',linewidths=1.1,zorder=5)
    ax.text(*pos[v],str(v),ha='center',va='center',fontsize=6.5 if len(str(v))<5 else 5.8,color='white',zorder=6)
ax.set(xlim=extent[:2],ylim=extent[2:],aspect='equal');ax.axis('off')
ax.set_title(f'{DATASET}: selected complete hyperedges\n{len(picture_vertices)} vertices / {n:,} total',fontsize=13)
tableax.axis('off');y=.95
tableax.text(0,y,'Exact displayed hyperedge memberships',weight='bold',fontsize=11);y-=.085
for number,j in enumerate(picture_edges):
    tableax.text(0,y,f'E{j+1}: '+', '.join(map(str,edges[j])),color=colors[number%10],fontsize=9);y-=.075
tableax.text(0,y-.025,f'Full-data communities: k={PRIMARY[0]}, s={PRIMARY[1]}',fontsize=10,weight='bold')
tableax.legend(handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=c,label=l,markersize=9)
    for c,l in [('#2b728b','One community'),('#c67419','Multiple communities'),('#9c9c9c','Unassigned')]],loc='lower left',frameon=False)
fig.text(.04,.035,'Contours = original hyperedges. Node colors = full-data membership status. Selected view is illustrative, not representative.',fontsize=9)
fig.subplots_adjust(bottom=.12,wspace=.12)
for ext in ['png','svg']:fig.savefig(OUT/('selected_hypergraph.'+ext),dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
pd.DataFrame({'hyperedge_id':[j+1 for j in picture_edges],'complete_members':[';'.join(map(str,edges[j])) for j in picture_edges]}).to_csv(OUT/'figure_hyperedges.csv',index=False)
pd.DataFrame({'node_id':picture_vertices,'label':[labels[v] for v in picture_vertices],
    'community_ids':[';'.join(map(str,memberships[index[v]])) for v in picture_vertices]}).to_csv(OUT/'figure_vertex_key.csv',index=False)



## 6. Export results, figures and runtime
PNG figures are 300 dpi; SVG figures are vector files. The final cell downloads a ZIP containing results, exact figure membership keys, full-data sensitivity, runtime and metadata.

Runtime reports single runs, not a scalability benchmark. Hyperedge-intersection work can still be quadratic in hyperedge count, even though blocking bounds intermediate memory. Pairwise projections cost up to Σ|e|² to construct. Naive exact vertex-isolation calculations require n all-pairs shortest-path computations; this is why Walmart uses an explicitly labelled subset. Further research validation needs planted communities, repeated timings and independent perturbation tests. No ground-truth recovery or modularity has been calculated here.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].bar([f'{r.k},{r.s}' for r in sensitivity.itertuples()],sensitivity.communities,color='#2b728b')
axes[0].set(xlabel='Parameters k,s',ylabel='Number of communities',title='Full-data parameter sensitivity')
sizes_sorted=sorted(map(len,communities),reverse=True)
axes[1].plot(range(1,len(sizes_sorted)+1),sizes_sorted,color='#2b728b')
axes[1].set(xlabel='Community rank',ylabel='Community size',yscale='log',title=f'Full-data communities: k={PRIMARY[0]}, s={PRIMARY[1]}')
fig.tight_layout()
for ext in ['png','svg']:fig.savefig(OUT/('community_overview.'+ext),dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6));ax.spy(H,markersize=.15 if DATASET=='Walmart' else .6,aspect='auto')
ax.set(xlabel='Unique hyperedge column',ylabel='Vertex row',title=DATASET+' full incidence sparsity')
fig.tight_layout()
for ext in ['png','svg']:fig.savefig(OUT/('incidence.'+ext),dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
corr=results[['Hyperdegree','Closeness','Betweenness','Eigenvector','Dangling']].corr(method='spearman')
corr.to_csv(OUT/'centrality_correlations.csv')
pd.DataFrame(timings).to_csv(OUT/'runtime.csv',index=False)
metadata={'dataset':DATASET,'sha256':archive_hash,'vertices':n,'records':len(raw),'unique_hyperedges':len(edges),
    'primary':PRIMARY,'settings':SETTINGS,'centrality_scope':scope,'centrality_vertices':nn,
    'analysis_hyperedges':len(analysis_edges),'community_scope':'full unique-unordered-set hypergraph',
    'seed':SEED,'versions':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'networkx':nx.__version__}}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
(OUT/'README.txt').write_text('Communities: full unique-unordered-set hypergraph.\nCentralities: '+scope+
    '\nThe Walmart subset is degree-biased and is not representative; its dangling values are not full-network scores.\n'
    'Distances and node-isolation dangling are computed on the unweighted two-section.\n'
    'Figures show selected COMPLETE original hyperedges, with holes excluding nonmembers. Colors show membership status, not community identity.\n'
    'Community percolation adopted from https://doi.org/10.1038/s41598-025-19974-9 .\n'
    'No modularity, ground-truth recovery, or clinical/causal claims are made.\n')
zip_path=Path(DATASET+'_Community_Results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
print('Saved',zip_path)
try:from google.colab import files
except ImportError:pass
else:files.download(str(zip_path))
